In [ ]:
%load_ext autoreload
%autoreload 2
from datetime import date
from athletics_performance import Athlete, Event, Performance, PerformanceCatalogue

# PerformanceCatalogue

`PerformanceCatalogue` groups a collection of `Performance` objects and exposes operations for:

- **Filtering** — by athlete, event, date range, season, category, club
- **Records & personal bests** — best performance in the catalogue, per athlete, per event
- **Ranking** — sort by result, extract top N, build a ranking table
- **Grouping** — split into sub-catalogues by any attribute
- **Statistics** — count, mean, median, stdev, min, max

All query methods return a **new** `PerformanceCatalogue`, so operations can be chained.

## Sample data

We set up four athletes, two events (100m and long jump), and a batch of performances spread across two seasons.

In [ ]:
# Athletes
alice = Athlete(
    licence="1001", last_name="Moreau", first_name="Alice", yob=1998, sex="F"
)
bob = Athlete(licence="1002", last_name="Martin", first_name="Bob", yob=1995, sex="M")
claire = Athlete(
    licence="1003", last_name="Dupont", first_name="Claire", yob=2000, sex="F"
)
david = Athlete(
    licence="1004", last_name="Berger", first_name="David", yob=1993, sex="M"
)

# Events
event_100m = Event(event_id="100m", name="100 mètres", measurement="time", unit="s")
event_lj = Event(
    event_id="LJ", name="Saut en longueur", measurement="distance", unit="m"
)


def p(perf_id, athlete, event, result, perf_date, category=None, club_id=None):
    """Shorthand factory used only in this notebook."""
    return Performance(
        perf_id=perf_id,
        date=perf_date,
        result_value=result,
        measurement=event.measurement,
        unit=event.unit,
        athlete=athlete,
        event=event,
        category_snapshot=category,
        club_id_snapshot=club_id,
    )


# 100m performances — season 2025 and 2026
perfs_100m = [
    p("T01", alice, event_100m, 13.10, date(2025, 5, 10), "SEF", "069069"),
    p("T02", alice, event_100m, 12.85, date(2025, 6, 20), "SEF", "069069"),
    p("T03", alice, event_100m, 12.60, date(2026, 3, 15), "SEF", "069069"),
    p("T04", alice, event_100m, 12.45, date(2026, 5, 22), "SEF", "069069"),
    p("T05", claire, event_100m, 13.50, date(2025, 5, 10), "ESF", "069069"),
    p("T06", claire, event_100m, 13.20, date(2026, 4, 18), "ESF", "069069"),
    p("T07", claire, event_100m, 12.90, date(2026, 6, 5), "ESF", "069069"),
    p("T08", bob, event_100m, 11.40, date(2025, 5, 10), "SEM", "069106"),
    p("T09", bob, event_100m, 11.20, date(2025, 7, 12), "SEM", "069106"),
    p("T10", bob, event_100m, 10.95, date(2026, 4, 25), "SEM", "069106"),
    p("T11", david, event_100m, 11.80, date(2025, 6, 1), "SEM", "069106"),
    p("T12", david, event_100m, 11.55, date(2026, 5, 30), "SEM", "069106"),
]

# Long jump performances
perfs_lj = [
    p("J01", alice, event_lj, 5.20, date(2025, 5, 10), "SEF", "069069"),
    p("J02", alice, event_lj, 5.45, date(2026, 3, 15), "SEF", "069069"),
    p("J03", alice, event_lj, 5.60, date(2026, 5, 22), "SEF", "069069"),
    p("J04", bob, event_lj, 6.80, date(2025, 5, 10), "SEM", "069106"),
    p("J05", bob, event_lj, 7.05, date(2026, 4, 25), "SEM", "069106"),
    p("J06", david, event_lj, 6.40, date(2025, 6, 1), "SEM", "069106"),
    p("J07", david, event_lj, 6.65, date(2026, 5, 30), "SEM", "069106"),
]

cat = PerformanceCatalogue(perfs_100m + perfs_lj)
print(cat)
print(f"Events covered: {set(p.event_id for p in cat)}")
print(f"Athletes:       {set(p.athlete_id for p in cat)}")

## Building a catalogue

In [ ]:
# Start empty and add incrementally
cat2 = PerformanceCatalogue()
cat2.add(perfs_100m[0])
cat2.extend(perfs_100m[1:3])
print("Built incrementally:", cat2)

# Concatenate two catalogues
cat_100m = PerformanceCatalogue(perfs_100m)
cat_lj = PerformanceCatalogue(perfs_lj)
merged = cat_100m + cat_lj
print("Merged (+ operator):", merged)

## Filtering

In [ ]:
# By athlete
alice_perfs = cat.filter(athlete_id=alice.licence)
print(f"Alice's performances : {len(alice_perfs)}")

# By event
all_100m = cat.filter(event_id="100m")
print(f"100m performances    : {len(all_100m)}")

# By season
season_2026 = cat.filter(yos=2026)
print(f"Season 2026          : {len(season_2026)} performances")

# By date range
spring_2026 = cat.filter(date_from=date(2026, 3, 1), date_to=date(2026, 5, 31))
print(f"Spring 2026          : {len(spring_2026)} performances")

# By category
sef_perfs = cat.filter(category="SEF")
print(f"SEF category         : {len(sef_perfs)} performances")

# Combined: Alice's 100m in season 2026
alice_100m_2026 = cat.filter(athlete_id=alice.licence, event_id="100m", yos=2026)
print(f"Alice / 100m / 2026  : {len(alice_100m_2026)} performances")
for pf in alice_100m_2026:
    print(f"  {pf.date}  {pf.result_value}s")

## Records and personal bests

In [ ]:
# Overall 100m record in the catalogue
rec_100m = cat.filter(event_id="100m").record()
print(
    f"100m catalogue record : {rec_100m.result_value}s  (athlete {rec_100m.athlete_id}, {rec_100m.date})"
)

# Long jump record
rec_lj = cat.filter(event_id="LJ").record()
print(
    f"LJ catalogue record   : {rec_lj.result_value}m  (athlete {rec_lj.athlete_id}, {rec_lj.date})"
)

# Personal best for a specific athlete on a specific event
pb = cat.filter(event_id="100m").personal_best(alice.licence)
print(f"\nAlice PB 100m : {pb.result_value}s  ({pb.date})")

# All personal bests (one per athlete × event)
print("\nAll personal bests:")
print(f"  {'athlete':<8} {'event':<6} {'result'}")
for pf in cat.personal_bests():
    unit = pf.unit
    print(f"  {pf.athlete_id:<8} {pf.event_id:<6} {pf.result_value}{unit}")

In [ ]:
# Records (best per event) within a given season
print("Season 2026 records:")
for pf in cat.filter(yos=2026).records():
    print(
        f"  {pf.event_id:<6} {pf.result_value}{pf.unit}  athlete={pf.athlete_id}  date={pf.date}"
    )

## Ranking

In [ ]:
# Season 2026 100m ranking (one PB per athlete, sorted best first)
ranking_100m_2026 = cat.filter(event_id="100m", yos=2026).ranking()

print("100m ranking — season 2026")
print(f"  {'pos':<4} {'athlete':<8} {'result':>8}")
for pos, pf in enumerate(ranking_100m_2026, start=1):
    print(f"  {pos:<4} {pf.athlete_id:<8} {pf.result_value:>7.2f}s")

print()

# Top 3 individual 100m performances ever in the catalogue
top3 = cat.filter(event_id="100m").top(3)
print("Top 3 individual 100m performances (all time):")
for pf in top3:
    print(f"  {pf.result_value:.2f}s  athlete={pf.athlete_id}  date={pf.date}")

## Grouping

In [ ]:
# Group by athlete — how many performances each athlete has
by_athlete = cat.group_by("athlete_id")
print("Performances per athlete:")
for athlete_id, sub in sorted(by_athlete.items()):
    print(f"  {athlete_id} → {len(sub)} performances")

In [ ]:
# Group by season — compare record progression year over year
print("100m record progression by season:")
by_yos = cat.filter(event_id="100m").group_by("yos")
for yos_key in sorted(by_yos):
    rec = by_yos[yos_key].record()
    print(f"  {yos_key}: {rec.result_value:.2f}s  (athlete {rec.athlete_id})")

In [ ]:
# Group by category — ranking per category for a single event and season
print("100m season-2026 ranking by category:")
by_cat = cat.filter(event_id="100m", yos=2026).group_by("category_snapshot")
for cat_name, sub in sorted(by_cat.items()):
    ranking = sub.ranking()
    print(f"\n  [{cat_name}]")
    for pos, pf in enumerate(ranking, start=1):
        print(f"    {pos}. athlete={pf.athlete_id}  {pf.result_value:.2f}s")

## Statistics

In [ ]:
# Stats on all 100m performances
s = cat.filter(event_id="100m").stats()
print("100m overall stats:")
for key, val in s.items():
    print(f"  {key:<8}: {val:.4f}" if isinstance(val, float) else f"  {key:<8}: {val}")

print()

# Season-by-season stats for 100m
print("100m stats per season:")
for yos_key in ["2025", "2026"]:
    s = (
        cat.filter(event_id="100m")
        .group_by("yos")
        .get(yos_key, PerformanceCatalogue())
        .stats()
    )
    print(
        f"  {yos_key}: count={s.get('count',0)}  mean={s.get('mean', 'n/a'):.2f}s  best={s.get('min', 'n/a'):.2f}s"
    )

## Composed queries

Because every method returns a new `PerformanceCatalogue`, complex analyses are just a chain of calls.

In [ ]:
# Athlete progression: best 100m per season for Alice
print(f"Alice's 100m progression ({alice.full_name}):")
alice_100m = cat.filter(athlete_id=alice.licence, event_id="100m")
for yos_key, sub in sorted(alice_100m.group_by("yos").items()):
    best = sub.record()
    print(f"  Season {yos_key}: {best.result_value:.2f}s  ({best.date})")

In [ ]:
# Club record for 100m (only athletes from club 069069)
club_rec = cat.filter(event_id="100m", club_id="069069").record()
print(
    f"Club 069069 — 100m record: {club_rec.result_value:.2f}s  (athlete {club_rec.athlete_id}, {club_rec.date})"
)

In [ ]:
# All-time personal bests per athlete across all events, sorted by event then result
print("All-time personal bests across all events:")
print(f"  {'athlete':<8} {'event':<6} {'result'}")
pb_all = cat.personal_bests()
for pf in sorted(pb_all, key=lambda x: (x.event_id, x.result_value)):
    print(f"  {pf.athlete_id:<8} {pf.event_id:<6} {pf.result_value}{pf.unit}")